# Module 13 Lab - Building ML Pipelines

**Objective:** To understand the importance of `scikit-learn` **Pipelines** for creating robust, reproducible, and professional machine learning workflows.

**In this lab, you will refactor code from a previous lab into a clean, professional `Pipeline` object.**

## Part 1: Why Use Pipelines?

**Concept:** As you've seen, a typical ML workflow involves multiple steps: loading data, cleaning it, splitting it, preprocessing features (scaling, encoding), and finally, training a model. Managing all these steps separately can be messy and error-prone.

**Data Leakage:** A major risk of manual preprocessing is **data leakage**. This happens when information from the test set accidentally "leaks" into the training process. For example, if you calculate the mean for scaling using the *entire* dataset before splitting, the model has already "seen" the test data, leading to overly optimistic performance estimates.

**A `scikit-learn` Pipeline solves these problems by:**
1.  **Encapsulating** all workflow steps into a single object.
2.  **Preventing Data Leakage:** It ensures that preprocessing steps are fitted *only* on the training data during cross-validation or when calling `.fit()`.
3.  **Improving Reproducibility:** The entire workflow is saved as one object, making it easy to reuse and deploy.

## Part 2: The "Manual" Way (What We Did Before)

Let's revisit the Titanic dataset and the steps we took to prepare the data and train a model. This code should look familiar. Notice how many separate objects and steps there are.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load data
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Basic feature engineering and cleaning
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df.drop('Cabin', axis=1, inplace=True)

X = df.drop(['Survived', 'Name', 'Ticket', 'PassengerId'], axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify feature types
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_features = ['Pclass', 'Sex', 'Embarked']

# Manual Preprocessing
scaler = StandardScaler()
X_train_scaled_num = scaler.fit_transform(X_train[numeric_features])
X_test_scaled_num = scaler.transform(X_test[numeric_features]) # Note: using .transform() here!

encoder = OneHotEncoder(handle_unknown='ignore')
X_train_encoded_cat = encoder.fit_transform(X_train[categorical_features])
X_test_encoded_cat = encoder.transform(X_test[categorical_features])

# Combine preprocessed features
X_train_processed = np.hstack((X_train_scaled_num, X_train_encoded_cat.toarray()))
X_test_processed = np.hstack((X_test_scaled_num, X_test_encoded_cat.toarray()))

# Train model
model = RandomForestClassifier(random_state=42)
model.fit(X_train_processed, y_train)
y_pred = model.predict(X_test_processed)

print(f"Accuracy (Manual Method): {accuracy_score(y_test, y_pred):.2%}")

/tmp/ipykernel_7868/1388338196.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipykernel_7868/1388338196.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

Accuracy (Manual Method): 82.68%


## Part 3: The "Pipeline" Way

Now, let's do the exact same thing but encapsulate all the preprocessing steps into a single `Pipeline`.

**Your Task:** Use `make_pipeline` and `make_column_transformer` to build a complete workflow. This is the modern, professional way to build models in `scikit-learn`.

In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

# Reload the data to start fresh
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
df.drop(['Cabin', 'Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)
X = df.drop('Survived', axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- ENTER YOUR CODE HERE ---

# 1. Create a pipeline for numeric features
#    This pipeline will first impute missing 'Age' values with the median, then scale the features.
numeric_transformer = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

# 2. Create a pipeline for categorical features
#    This pipeline will first impute missing 'Embarked' values with the most frequent value, then one-hot encode.
categorical_transformer = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore')
)

# 3. Use ColumnTransformer to apply different transformers to different columns
preprocessor = make_column_transformer(
    (numeric_transformer, ['Age', 'Fare', 'SibSp', 'Parch']),
    (categorical_transformer, ['Pclass', 'Sex', 'Embarked'])
)

# 4. Create the final, full pipeline
#    This chains the preprocessor and the final model together.
final_pipeline = make_pipeline(
    preprocessor,
    RandomForestClassifier(random_state=42)
)

# 5. Fit and evaluate the entire pipeline in one step!
final_pipeline.fit(X_train, y_train)
y_pred_pipeline = final_pipeline.predict(X_test)

print(f"Accuracy (Pipeline Method): {accuracy_score(y_test, y_pred_pipeline):.2%}")

Accuracy (Pipeline Method): 82.68%


## 📝 Reflective Knowledge Check

**Instructions:** Answer the following questions in this markdown cell.

1.  **Code Comparison:** Look at the "Manual Way" versus the "Pipeline Way". What are the three biggest advantages you see in using the Pipeline approach?

2.  **Data Leakage Explained:** In the manual code, we used `scaler.fit_transform()` on the training data but only `scaler.transform()` on the test data. Why was this distinction crucial? How does the Pipeline automatically handle this for you?

3.  **Extending the Pipeline:** Imagine you wanted to add a PCA step to reduce dimensionality *after* scaling and encoding but *before* the RandomForestClassifier. How would you modify your `final_pipeline` object to include this step? (You don't need to write the full code, just describe where you would add `PCA()`.)

4.  **Real-World Value:** You are handing your model over to another team to deploy into a web application. Why is giving them the single `final_pipeline` object much safer and more reliable than giving them the 5 separate objects (`scaler`, `encoder`, `model`, etc.) from the manual approach?

**[ENTER YOUR ANSWERS HERE]**

1. One of the advantages of using a Pipeline is that it can reduce mistakes because there is less code to wrote, yet is clearer. It also prevents data leakage. The pipeline makes sure that the model only sees the training data. Last but not least, the other advantage of it is that the code is clearer, more understandable, and easy to explain.

2. In the manual code, we used fit_transform() on the training data because the scaler needed to learn the mean and standard deviation from the training set. For the test data, we used only transform() because the test data should not be used to learn scaling rules. This is important because using the test data during preprocessing would cause data leakage and make the model look more accurate that it really is. The Pipeline handles this automatically by fitting preprocessing steps only on the training data during .fit() and then applying those same transformations to the test data during .predict().

3. I would add the PCA step after the preprocessor and before the RandomForestClassifier. This means the data would first get cleaned, scaled, and encoded. Then PCA would reduce the number of features. After that, the Random Forest model would training using those features.

4. Giving another team the single final_pipeline object is safer because it includes all preprocessing steps and the trained model together. If we give them those objects separately, they might forget a step, apply them in the wrong order, or process new data differently from the training data. However, with the pipeline, the new data goes through the same exact process automatically making it more reliable with less errors.